# DiffBind Analysis — YabLab CUT&RUN

Template notebook for occupancy + affinity differential binding analysis.
Built to be executed by the Snakemake `diffbind_notebook` rule (via `papermill`),
or modified and re-run interactively.

Pipeline outputs consumed:
- Per-replicate filtered BAMs (`Bowtie2/target/*.filt.bam`)
- Pooled control BAM (`pooled_controls/*.pooled.bam`)
- Per-replicate peak calls (MACS3 narrowPeak or SEACR bed)
- DiffBind samplesheet (`DiffBind/samplesheet.csv`)


## Parameters

Overridden by `papermill` when run from Snakemake.


In [ ]:
# Papermill injects overrides for these via the tagged parameters cell.
samplesheet_path <- "OUTPUT/DiffBind/samplesheet.csv"
outdir           <- "OUTPUT/DiffBind"
min_overlap      <- 2
fdr_threshold    <- 0.05
summit_width     <- 250        # half-width around each peak summit for read counting
score_method     <- "DBA_SCORE_NORMALIZED"   # see ?dba.count


## Setup


In [ ]:
suppressPackageStartupMessages({
    library(DiffBind)
    library(tidyverse)
})
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)
sessionInfo()

## Load the samplesheet

The samplesheet is generated by the `diffbind_samplesheet` rule from the pipeline config.
Columns: SampleID, Tissue, Factor, Condition, Treatment, Replicate, bamReads,
ControlID, bamControl, Peaks, PeakCaller.


In [ ]:
samples <- read.csv(samplesheet_path, stringsAsFactors = FALSE)
stopifnot(nrow(samples) > 0)
samples

## Build the DBA object (peakset / occupancy view)


In [ ]:
dba_obj <- dba(sampleSheet = samples, minOverlap = min_overlap)
dba_obj

## Occupancy analysis

Occupancy mode uses presence/absence of peaks across samples — useful for
reproducibility checks and consensus peakset construction.


In [ ]:
# Peak overlap rate: how many peaks reproduce across N replicates
olap_rate <- dba.overlap(dba_obj, mode = DBA_OLAP_RATE)
olap_rate

pdf(file.path(outdir, "occupancy_overlap_rate.pdf"), width = 6, height = 4)
plot(olap_rate, type = "b", ylab = "# peaks", xlab = "Overlap in at least this many samples")
dev.off()


In [ ]:
pdf(file.path(outdir, "occupancy_heatmap.pdf"), width = 8, height = 8)
dba.plotHeatmap(dba_obj, colScheme = "Reds")
dev.off()

pdf(file.path(outdir, "occupancy_pca.pdf"), width = 6, height = 6)
dba.plotPCA(dba_obj, attributes = DBA_CONDITION, label = DBA_ID)
dev.off()


In [ ]:
# Export consensus peaks (those passing minOverlap)
consensus_gr <- dba.peakset(dba_obj, bRetrieve = TRUE, minOverlap = min_overlap)
consensus_df <- as.data.frame(consensus_gr)
write.table(
    consensus_df[, c("seqnames", "start", "end")],
    file = file.path(outdir, "consensus_peaks.bed"),
    sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE
)
message(sprintf("Consensus peaks (minOverlap=%d): %d", min_overlap, nrow(consensus_df)))


## Count reads in the consensus peakset

`dba.count` recenters peaks on summits, builds a fixed-width region around each,
and counts reads per sample → affinity matrix.


In [ ]:
dba_counts <- dba.count(
    dba_obj,
    minOverlap = min_overlap,
    summits    = summit_width,
    bUseSummarizeOverlaps = TRUE
)
dba_counts


## Normalize


In [ ]:
dba_norm <- dba.normalize(dba_counts)
dba.normalize(dba_norm, bRetrieve = TRUE)


## Define contrasts and run differential analysis

By default, contrast across the `Condition` factor (one contrast per pair of conditions
with ≥ `minMembers` replicates).


In [ ]:
dba_con <- dba.contrast(dba_norm, categories = DBA_CONDITION, minMembers = 2)
dba.show(dba_con, bContrasts = TRUE)


In [ ]:
dba_res <- dba.analyze(dba_con)
contrast_tbl <- dba.show(dba_res, bContrasts = TRUE)
contrast_tbl


## Affinity-level plots (post-counting)


In [ ]:
pdf(file.path(outdir, "affinity_pca.pdf"), width = 6, height = 6)
dba.plotPCA(dba_res, attributes = DBA_CONDITION, label = DBA_ID)
dev.off()

pdf(file.path(outdir, "affinity_heatmap_correlation.pdf"), width = 8, height = 8)
dba.plotHeatmap(dba_res, correlations = TRUE)
dev.off()


## Per-contrast results: report tables + MA + volcano + binding-affinity heatmap


In [ ]:
n_contrasts <- nrow(contrast_tbl)
if (n_contrasts == 0) {
    message("No contrasts available — need ≥2 conditions with ≥2 replicates each.")
} else {
    for (i in seq_len(n_contrasts)) {
        cname <- paste0(
            contrast_tbl$Group[i], "_vs_", contrast_tbl$Group2[i]
        )
        message(sprintf("Contrast %d: %s", i, cname))

        # Full report (all sites, with stats)
        rep_all <- dba.report(dba_res, contrast = i, th = 1)
        rep_df  <- as.data.frame(rep_all)
        write.csv(
            rep_df,
            file.path(outdir, sprintf("contrast_%02d_%s_results.csv", i, cname)),
            row.names = FALSE
        )

        # Significant sites only
        rep_sig <- dba.report(dba_res, contrast = i, th = fdr_threshold)
        sig_df  <- as.data.frame(rep_sig)
        write.csv(
            sig_df,
            file.path(outdir, sprintf("contrast_%02d_%s_significant_FDR%g.csv",
                                       i, cname, fdr_threshold)),
            row.names = FALSE
        )
        message(sprintf("  significant @ FDR<%g: %d / %d",
                        fdr_threshold, nrow(sig_df), nrow(rep_df)))

        pdf(file.path(outdir, sprintf("contrast_%02d_%s_MA.pdf", i, cname)),
            width = 6, height = 5)
        dba.plotMA(dba_res, contrast = i, th = fdr_threshold)
        dev.off()

        pdf(file.path(outdir, sprintf("contrast_%02d_%s_volcano.pdf", i, cname)),
            width = 6, height = 5)
        dba.plotVolcano(dba_res, contrast = i, th = fdr_threshold)
        dev.off()

        # Binding-affinity heatmap of significant sites (skip if too few)
        if (nrow(sig_df) >= 2) {
            pdf(file.path(outdir, sprintf("contrast_%02d_%s_affinity_sig_heatmap.pdf",
                                          i, cname)),
                width = 8, height = 8)
            tryCatch(
                dba.plotHeatmap(dba_res, contrast = i, correlations = FALSE,
                                scale = "row", th = fdr_threshold),
                error = function(e) message("Heatmap skipped: ", conditionMessage(e))
            )
            dev.off()
        }
    }
}


## Save the full DBA object

Saved as RDS for downstream re-analysis without re-counting reads.


In [ ]:
saveRDS(dba_res, file = file.path(outdir, "dba_analysis.rds"))
message("Wrote: ", file.path(outdir, "dba_analysis.rds"))
